# Fase 0-1: Problema, contexto y diseño técnico

**Gold Price Prediction** - Serie temporal financiera (forecasting de regresión).

## 0. Problema y contexto

### 0.1 Contexto

- **Problema de negocio:** estimar el precio spot del oro (*gold_spot*, USD/oz) a 1, 5 y 21 días hábiles vista.
- **Usuario final:** analista de mercados / gestor de cartera que necesita una referencia cuantitativa de precios futuros.
- **Proceso actual sin automatización:** heurísticas, reglas técnicas (soportes/resistencias) y juicio de analistas.
- **Decisión que apoya:** sizing de posiciones, timing de compra/venta, *stress testing* de carteras y alertas.
- **Valor esperado:** referencia objetiva + métricas de incertidumbre; **coste**: los errores de predicción se traducen en P&L, pero este es un proyecto educativo/analítico (sin trading automático).
- **Alternativas no ML:** *naive* (último precio), media móvil, ARIMA.

### 0.2 Formulación técnica

- **Tipo:** Forecasting de regresión univariante con covariables exógenas (múltiples horizontes).
- **Unidad de predicción:** día hábil (mercado COMEX/NYMEX).
- **Entrada disponible en el momento de predecir `t`:** precio del oro y 59 exógenas en `t` (y pasadas) - nunca futuras.
- **Salida esperada:** precio del oro en `t+h`.
- **Horizonte:** h en {1, 5, 21} días hábiles. Frecuencia de inferencia: diaria (batch).
- **Límites:** latencia < 1 s por predicción, memoria < 2 GB, interpretabilidad media-alta (SHAP).

## 1. Diseño técnico y reproducibilidad

### 1.1 Estructura del proyecto

```
gold-price-prediction-v2/
|-- configs/            # config.yaml (rutas, split, features) + params.yaml (hiperparámetros)
|-- data/raw|interim|processed/
|-- notebooks/          # 01_... a 23_... (una fase por notebook)
|-- src/                # data, features, models, evaluation, api
|-- models/             # artefactos .joblib
|-- reports/figures/    # gráficos y métricas
|-- tests/              # pytest
+---- docs/               # model card, informe técnico, manual API
```

### 1.2 Reproducibilidad

- Semilla global 42 (`src/utils.set_seed`).
- Python 3.11.9, dependencias fijadas en `requirements.txt`.
- Dataset versionado: `data/raw/gold-price-prediction-dataset.csv` (inmutable).
- Configuración de split y features en `configs/config.yaml` (separada del código).
- Experimentos registrados en `reports/` (JSON + figuras).

## Resumen de la fase

| Concepto | Valor |
|---|---|
| Problema | Regresión de precio spot del oro a 1/5/21 días |
| Unidad | Día hábil |
| Métrica primaria | RMSE / MAE en USD, sMAPE |
| Split | Temporal estricto (train 2000-2019, val 2020-2022, test 2023-2025) |
| Entorno | Python 3.11 + scikit-learn/XGBoost/LightGBM/CatBoost |

---


Configuración del notebook: rutas raíz y semilla global.

Este bloque se repite en todos los notebooks para que puedan ejecutarse
desde cualquier directorio de trabajo (Jupyter, nbconvert, VS Code...).

In [1]:
import sys
from pathlib import Path

def _find_root():
    p = Path.cwd()
    for _ in range(5):
        if (p / "configs" / "config.yaml").exists():
            return p
        p = p.parent
    return Path.cwd()

ROOT = _find_root()
sys.path.insert(0, str(ROOT))

from src.utils import set_publication_style, set_seed
set_seed(42)
set_publication_style()

print(f"Raíz del proyecto: {ROOT}")

Raíz del proyecto: C:\Users\sgml1\Desktop\gold-price-prediction-v2


Verificación del entorno: versiones de Python y librerías clave.

Comprobamos que el entorno es el esperado (Python 3.11+, pandas, numpy,
scikit-learn) antes de empezar. Si algo falla aquí, hay que revisar
`requirements.txt` y el virtualenv.

In [2]:
import sys

import numpy as np
import pandas as pd
import sklearn

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__, "| numpy:", np.__version__, "| sklearn:", sklearn.__version__)

Python: 3.11.9
pandas: 3.0.5 | numpy: 2.4.6 | sklearn: 1.9.0


Carga de la configuración global del proyecto.

`configs/config.yaml` centraliza rutas, ventana temporal, split y features;
`configs/params.yaml` guarda los hiperparámetros (resultado de la fase 14).
Así el código no tiene valores mágicos y todo es reproducible.

In [3]:
from src.config import get_config, get_params

cfg = get_config()
params = get_params()

print("Ventana de datos:", cfg["data"]["start_date"], "->", cfg["data"]["end_date"])
print("Split temporal:", cfg["split"])
print("Horizontes de predicción:", cfg["target"]["horizons"])
print("Modelos configurados:", list(params["models"].keys()))
print("Modelo seleccionado:", params.get("model_selection", {}).get("selected", "-"))

Ventana de datos: 2000-01-01 -> 2025-09-12
Split temporal: {'train': ['2000-01-01', '2019-12-31'], 'val': ['2020-01-01', '2022-12-31'], 'test': ['2023-01-01', '2025-09-12']}
Horizontes de predicción: [1, 5, 21]
Modelos configurados: ['ridge', 'random_forest', 'xgboost', 'lightgbm', 'catboost', 'mlp']
Modelo seleccionado: ridge
